# 01 · Data inventory

This notebook catalogs the data for the Index Life-Cycle Strategy and builds a **provisional register** of effective S&P 500 additions. The register feeds the event study and the post-inclusion portfolio.

Place the original files in `Data/` at the repository root. Run the notebook from the root or the `notebooks/` directory using the project's Python kernel with `pandas` and `openpyxl` installed. Results stay in memory. Clear cell outputs before sharing the notebook to keep WRDS data local.

In [ ]:
from pathlib import Path

import pandas as pd

project_root = next(
    (folder for folder in (Path.cwd(), *Path.cwd().parents)
     if (folder / "notebooks" / "01_inventory.ipynb").is_file()),
    None,
)
if project_root is None:
    raise FileNotFoundError("Open the notebook from the AppliedInvestments root or the notebooks directory.")

data_dir = project_root / "Data"
sources = pd.DataFrame([
    ("sp500_snapshot", "Index membership", "S&P 1975 Wharton Research Data Services.xlsx", "Initial index membership with PERMNO"),
    ("sp500_history", "Index membership", "sp500_constituent_changes.csv", "Historical membership spells"),
    ("sp500_additions", "Index membership", "sp500_additions_2023_2025.csv", "Recent additions by effective date"),
    ("sp500_removals", "Index membership", "sp500_removals_2023_2025.csv", "Recent removals with partial coverage"),
    ("crsp_names", "CRSP", "crsp_security_history.csv", "Historical tickers and security attributes"),
    ("crsp_monthly", "CRSP", "crsp_monthly_stock_returns.csv", "Monthly returns and market capitalizations"),
    ("crsp_daily", "CRSP", "crsp_daily_stock_returns_and_quotes.csv", "Daily returns, prices, volume, and bid and ask quotes"),
    ("crsp_delist", "CRSP", "crsp_delisting_events.csv", "Delisting dates, reasons, and returns"),
    ("index_monthly", "Benchmark", "crsp_monthly_index_returns.csv", "S&P 500 and CRSP market series, monthly"),
    ("index_daily", "Benchmark", "crsp_daily_index_returns.csv", "S&P 500 and CRSP market series, daily"),
    ("ccm_link", "Compustat", "crsp_compustat_link_history.csv", "Dated GVKEY–PERMNO link"),
    ("comp_quarterly", "Compustat", "compustat_quarterly_fundamentals.csv", "Quarterly financials"),
    ("comp_annual", "Compustat", "compustat_annual_fundamentals.csv", "Annual financials, including assets, earnings, and buybacks"),
], columns=["dataset", "category", "filename", "usage"]).set_index("dataset")

paths = {dataset: data_dir / filename for dataset, filename in sources["filename"].items()}
legacy_filenames = {
    "crsp_names": "honhxqxgerziffrf.csv",
    "crsp_monthly": "jhvzegjsguw3gw3k.csv",
    "crsp_daily": "wxqvlgh1rjhdvaei.csv",
    "crsp_delist": "zopuwel31lgljgik.csv",
    "index_monthly": "pc8rgx9sahq2tuh6.csv",
    "index_daily": "z8pjwhvfhit5vwei.csv",
    "ccm_link": "kg7lz7q7gsbmphju.csv",
    "comp_quarterly": "pqwyvdhxalbjomaa.csv",
    "comp_annual": "rdmfynls1xcgmjjb.csv",
}
for dataset, filename in legacy_filenames.items():
    legacy_path = data_dir / filename
    if not paths[dataset].is_file() and legacy_path.is_file():
        paths[dataset] = legacy_path

sources["local_filename"] = [path.name for path in paths.values()]
sources["available"] = [path.is_file() for path in paths.values()]
sources["MiB"] = [
    round(path.stat().st_size / 1024**2, 2) if path.is_file() else pd.NA
    for path in paths.values()
]
display(sources[["category", "local_filename", "usage", "available", "MiB"]])
missing_files = sources.loc[~sources["available"], "filename"].tolist()
if missing_files:
    raise FileNotFoundError(f"Missing files in {data_dir}. {', '.join(missing_files)}")

## Initial membership and historical records

The workbook provides the starting composition in early 1975. An `Added` row and a `Removed` row can describe the same membership spell. A spell is identified by `PERMNO, start, end`. The latest end date marks the coverage boundary of the historical export.

In [ ]:
sample_start = pd.Timestamp("1975-01-01")
snapshot = pd.read_excel(paths["sp500_snapshot"], header=1)
for column in ["SP500 Start", "SP500 End"]:
    snapshot[column] = pd.to_datetime(
        snapshot[column].str.replace(".", "", regex=False), format="mixed"
    )

history = pd.read_csv(paths["sp500_history"], parse_dates=["SP500 Start", "SP500 End"])
history = history.rename(columns={
    "PERMNO": "permno", "Ticker": "ticker", "Company": "company",
    "SP500 Start": "start", "SP500 End": "end",
})
spell_key = ["permno", "start", "end"]
historical_spells = history.drop_duplicates(spell_key).sort_values(spell_key)
history_end = historical_spells["end"].max()
snapshot_ids = set(snapshot.loc[
    snapshot["SP500 Start"].le(sample_start) & snapshot["SP500 End"].ge(sample_start), "PERMNO"
])
historical_start_ids = set(historical_spells.loc[
    historical_spells["start"].le(sample_start) & historical_spells["end"].ge(sample_start), "permno"
])
previous_end = historical_spells.groupby("permno")["end"].shift()
display(pd.Series({
    "Securities in the workbook at the start of 1975": len(snapshot_ids),
    "Securities in the historical export at that date": len(historical_start_ids),
    "Workbook securities missing from the historical export at that date": len(snapshot_ids - historical_start_ids),
    "Rows in the historical export": len(history),
    "Distinct membership spells": len(historical_spells),
    "Overlapping consecutive spells for the same security": int(
        previous_end.ge(historical_spells["start"]).sum()
    ),
    "End of historical coverage": history_end.date(),
}, name="check"))

## Match recent additions and removals

A ticker matches a PERMNO when the CRSP ticker validity interval covers the effective date. The public S&P notice seed supplements the supplied additions when a notice identifies an omitted company. For removals, a window of ±10 days identifies potential matches when a ticker stops trading near the effective date. The diagnostic shows how many matches remain ambiguous.

In [ ]:
names = pd.read_csv(
    paths["crsp_names"],
    usecols=["permno", "ticker", "secinfostartdt", "secinfoenddt"],
    parse_dates=["secinfostartdt", "secinfoenddt"],
).dropna(subset=["ticker"])
names["ticker"] = names["ticker"].str.upper().str.strip()

recent_additions = pd.read_csv(
    paths["sp500_additions"], parse_dates=["effective_date", "announcement_date"]
)
recent_removals = pd.read_csv(paths["sp500_removals"], parse_dates=["effective_date"])

notice_key = ["ticker", "effective_date"]
public_notices = pd.read_csv(
    project_root / "references" / "sp500_announcement_seed.csv",
    parse_dates=["effective_date", "announcement_date", "midcap400_exit_date", "narrative_replacement_date"],
)
if public_notices.duplicated(notice_key).any():
    raise ValueError("The S&P notice seed contains a repeated ticker and effective date.")
supplements = public_notices.loc[public_notices["company"].notna(), [
    *notice_key, "announcement_date", "company", "event_type",
]]
missing_locally = ~pd.MultiIndex.from_frame(supplements[notice_key]).isin(
    pd.MultiIndex.from_frame(recent_additions[notice_key])
)
recent_additions = pd.concat([recent_additions, supplements.loc[missing_locally].assign(
    source="sp_dji_notice", migration_from_midcap400="unknown",
)], ignore_index=True)


def ticker_candidates(events, tolerance_days=0):
    keys = events[["ticker", "effective_date"]].rename_axis("event_id").reset_index()
    keys["ticker"] = keys["ticker"].str.upper().str.strip()
    matches = keys.merge(names, on="ticker", how="left")
    tolerance = pd.Timedelta(days=tolerance_days)
    valid = matches["effective_date"].between(
        matches["secinfostartdt"] - tolerance, matches["secinfoenddt"] + tolerance
    )
    return matches.loc[valid, ["event_id", "permno"]].drop_duplicates()


def candidate_counts(events, matches):
    return matches.groupby("event_id")["permno"].size().reindex(events.index, fill_value=0)


add_matches = ticker_candidates(recent_additions)
add_counts = candidate_counts(recent_additions, add_matches)
unique_add_matches = add_matches.loc[add_matches["event_id"].isin(add_counts[add_counts.eq(1)].index)]
recent_additions["permno"] = recent_additions.index.to_series().map(
    unique_add_matches.set_index("event_id")["permno"]
).astype("Int64")

removal_counts = candidate_counts(recent_removals, ticker_candidates(recent_removals))
removal_counts_with_tolerance = candidate_counts(
    recent_removals, ticker_candidates(recent_removals, tolerance_days=10)
)
match_counts = pd.DataFrame({
    "Additions, exact date": add_counts.value_counts(),
    "Removals, exact date": removal_counts.value_counts(),
    "Removals, ±10 days": removal_counts_with_tolerance.value_counts(),
}).fillna(0).astype(int).sort_index()
display(match_counts.rename_axis("Candidate PERMNOs"))
display(pd.DataFrame({
    "Additions": recent_additions.groupby(recent_additions["effective_date"].dt.year).size(),
    "Removals": recent_removals.groupby(recent_removals["effective_date"].dt.year).size(),
}))

## Provisional additions register

Historical candidates are membership spells that start in 1975 or later. The initial index composition is tracked separately. Recent additions retain their source and supplied fields. Missing announcement dates remain blank, and the migration status remains `unknown` when the source provides no classification.

The twelve-month check measures calendar coverage after the inclusion month. Check stock return coverage and apply event filters when building the analysis sample.

In [ ]:
historical_additions = historical_spells.loc[
    historical_spells["start"].between(sample_start, history_end),
    ["permno", "ticker", "company", "start"],
].rename(columns={"start": "effective_date"}).assign(
    source="historical_membership_export",
    event_type="unknown",
    migration_from_midcap400="unknown",
    announcement_date=pd.NaT,
)
event_columns = [
    "permno", "ticker", "company", "effective_date", "announcement_date",
    "event_type", "migration_from_midcap400", "source",
]
events = pd.concat(
    [historical_additions[event_columns], recent_additions[event_columns]],
    ignore_index=True,
).sort_values(["effective_date", "permno"], ignore_index=True)
events["permno"] = events["permno"].astype("Int64")
resolved = events.loc[events["permno"].notna()]
if resolved.duplicated(["permno", "effective_date"]).any():
    raise ValueError("Multiple additions share the same PERMNO and effective date.")

calendar_dates = {}
for dataset, date_column in [("index_daily", "dlycaldt"), ("index_monthly", "mthcaldt")]:
    calendar_dates[dataset] = pd.read_csv(
        paths[dataset], usecols=[date_column], parse_dates=[date_column]
    )[date_column]
display(pd.DataFrame({
    dataset: {"first date": dates.min().date(), "last date": dates.max().date()}
    for dataset, dates in calendar_dates.items()
}).T)

last_month = calendar_dates["index_monthly"].max().to_period("M")
events["calendar_has_12_months"] = events["effective_date"].dt.to_period("M") + 12 <= last_month
events["period"] = pd.cut(
    events["effective_date"].dt.year,
    bins=[1974, 1999, 2009, 2017, 2025],
    labels=["1975–1999", "2000–2009", "2010–2017", "2018–2025"],
)
display(events.groupby("period", observed=True).agg(
    additions=("effective_date", "size"),
    with_permno=("permno", "count"),
    with_announcement_date=("announcement_date", "count"),
    reported_migrations=("migration_from_midcap400", lambda values: values.eq("yes").sum()),
    unknown_migration_status=("migration_from_midcap400", lambda values: values.eq("unknown").sum()),
    full_12_month_calendar=("calendar_has_12_months", "sum"),
))

## Optional audit of the monthly export

This cell reads about 371 MB. Set `audit_monthly` to `True` to run it. The audit compares market fields while omitting ticker and other redundant identifiers. After dropping identical records on those fields, it raises an error if a security and month still have conflicting observations.

In [ ]:
audit_monthly = False
if audit_monthly:
    monthly_key = ["PERMNO", "MthCalDt"]
    monthly_columns = monthly_key + ["MthPrc", "MthDelFlg", "MthCap", "MthRet", "MthRetx", "MthVol"]
    monthly_raw = pd.read_csv(
        paths["crsp_monthly"], usecols=monthly_columns, parse_dates=["MthCalDt"]
    )
    monthly_market = monthly_raw.drop_duplicates()
    if monthly_market.duplicated(monthly_key).any():
        raise ValueError("Conflicting market observations for the same security and month.")
    display(pd.Series({
        "Export rows": len(monthly_raw),
        "Duplicate rows on market fields": len(monthly_raw) - len(monthly_market),
        "Missing monthly returns after removing duplicates": int(monthly_market["MthRet"].isna().sum()),
        "First date": monthly_market["MthCalDt"].min().date(),
        "Last date": monthly_market["MthCalDt"].max().date(),
    }, name="check"))
    del monthly_raw

In [ ]:
from csv import reader
from io import TextIOWrapper
from zipfile import ZipFile


def read_monthly_factors(archive_path, columns):
    with ZipFile(archive_path) as archive:
        filename, = [name for name in archive.namelist() if name.lower().endswith(".csv")]
        with archive.open(filename) as file:
            rows = ([value.strip() for value in row] for row in reader(TextIOWrapper(file, encoding="utf-8-sig")))
            header = next(row for row in rows if set(columns).issubset(row))
            monthly_rows = [row for row in rows if row and len(row[0]) == 6 and row[0].isdigit()]

    factors = pd.DataFrame(monthly_rows, columns=["yyyymm", *header[1:]])
    factors["month"] = pd.to_datetime(factors.pop("yyyymm"), format="%Y%m").dt.to_period("M")
    values = factors.set_index("month")[columns].astype(float)
    return values.mask(values.isin([-99.99, -999])).div(100)

## Data to complete before the remaining tests

- Reconcile index membership using the initial workbook and historical spells. Check 2023–2025 additions and removals against S&P notices. The removals file ends in August 2025. The [December 5, 2025 announcement](https://www.spglobal.com/spdji/en/documents/indexnews/announcements/20251205-1480992/1480992_december2025rebalance.pdf) confirms three removals effective December 22. Treat the supplied fields as provisional.
- Collect announcement dates for the announcement windows and S&P MidCap 400 membership history to identify migrations. The post-inclusion prototype uses the market and risk-free series from the Kenneth French archive. The planned multifactor test requires the five Fama–French factors and momentum. Passive ownership data is needed for the proposed mechanism test.
- Validate events, return coverage, and delisting treatment when building the final analysis samples. [WRDS documentation on delisting returns in CIZ](https://wrds-www.wharton.upenn.edu/pages/wrds-research/macros/run-an-event-study-ciz-format-macro/).

In [ ]:
import numpy as np

panel_columns = {
    "PERMNO": "permno", "MthCalDt": "mthcaldt", "MthCap": "mthcap", "MthPrc": "mthprc",
    "MthRet": "mthret", "MthVol": "mthvol", "MthDelFlg": "mthdelflg",
}
monthly_panel = pd.read_csv(
    paths["crsp_monthly"], usecols=list(panel_columns), parse_dates=["MthCalDt"]
).rename(columns=panel_columns).drop_duplicates()
if monthly_panel.duplicated(["permno", "mthcaldt"]).any():
    raise ValueError("Conflicting monthly observations for the same security and month.")
monthly_panel = monthly_panel.sort_values(["permno", "mthcaldt"], ignore_index=True)
panel_rows = monthly_panel[["permno", "mthcaldt"]].reset_index()

# S&P 500 membership: workbook spells capped at the export boundary plus historical spells (inclusive).
panel_spells = pd.concat([
    historical_spells[["permno", "start", "end"]],
    snapshot.rename(columns={"PERMNO": "permno", "SP500 Start": "start", "SP500 End": "end"})
    [["permno", "start", "end"]].assign(end=lambda s: s["end"].clip(upper=history_end)),
], ignore_index=True).drop_duplicates()
spell_matches = panel_rows.merge(panel_spells, on="permno")
member_rows = spell_matches.loc[
    spell_matches["mthcaldt"].between(spell_matches["start"], spell_matches["end"]), "index"
].unique()
monthly_panel["in_sp500"] = monthly_panel.index.isin(member_rows)

# Eligibility as in notebook 03: U.S. ordinary common equity on NYSE, NYSE American, or Nasdaq.
security_terms = pd.read_csv(
    paths["crsp_names"],
    usecols=["permno", "secinfostartdt", "secinfoenddt", "usincflg",
             "securitytype", "securitysubtype", "sharetype", "primaryexch"],
    parse_dates=["secinfostartdt", "secinfoenddt"],
)
eligible_terms = security_terms.loc[
    security_terms["usincflg"].eq("Y") & security_terms["securitytype"].eq("EQTY")
    & security_terms["securitysubtype"].eq("COM") & security_terms["sharetype"].eq("NS")
    & security_terms["primaryexch"].isin(["N", "A", "Q"]),
    ["permno", "secinfostartdt", "secinfoenddt"],
].sort_values("secinfostartdt")
dated_terms = pd.merge_asof(
    panel_rows.sort_values("mthcaldt"), eligible_terms,
    left_on="mthcaldt", right_on="secinfostartdt", by="permno", direction="backward",
)
eligible_rows = dated_terms.loc[dated_terms["mthcaldt"].le(dated_terms["secinfoenddt"]), "index"]
monthly_panel["eligible"] = (
    monthly_panel.index.isin(eligible_rows)
    & monthly_panel["mthcap"].gt(0) & monthly_panel["mthdelflg"].eq("N")
)

# Contamination proxy: no CRSP row in the three calendar months before the effective month.
# Uses only rows dated before the effective month. Events too early to assess stay unflagged.
crsp_months = monthly_panel[["permno"]].assign(
    panel_month=monthly_panel["mthcaldt"].dt.year * 12 + monthly_panel["mthcaldt"].dt.month
)
event_month = events["effective_date"].dt.year * 12 + events["effective_date"].dt.month
event_rows = events.loc[events["permno"].notna(), ["permno"]].astype("int64").assign(
    event_month=event_month
).reset_index().merge(crsp_months, on="permno")
has_prior_row = event_rows.loc[
    (event_rows["event_month"] - event_rows["panel_month"]).between(1, 3), "index"
].unique()
events["contamination_assessable"] = event_month - 3 >= crsp_months["panel_month"].min()
events["likely_spin_or_merger"] = (
    events["permno"].notna() & events["contamination_assessable"] & ~events.index.isin(has_prior_row)
)
display(events.groupby(events["effective_date"].dt.year // 10 * 10).agg(
    additions=("effective_date", "size"),
    not_assessable=("contamination_assessable", lambda values: (~values).sum()),
    flagged_excluded=("likely_spin_or_merger", "sum"),
).rename_axis("decade"))

# Labels: an addition becomes effective within the next 3 or 6 months (missing if the horizon is incomplete).
label_targets = events.loc[
    events["permno"].notna() & events["effective_date"].le(history_end) & ~events["likely_spin_or_merger"],
    ["permno", "effective_date"]
].astype({"permno": "int64"}).sort_values("effective_date")
next_effective = pd.merge_asof(
    panel_rows.sort_values("mthcaldt"), label_targets,
    left_on="mthcaldt", right_on="effective_date", by="permno",
    direction="forward", allow_exact_matches=False,
).set_index("index")["effective_date"].reindex(monthly_panel.index)
panel_month = monthly_panel["mthcaldt"].dt.to_period("M")
for horizon in (3, 6):
    horizon_month = panel_month + horizon
    monthly_panel[f"y{horizon}"] = (
        next_effective.dt.to_period("M").le(horizon_month)
        .where(horizon_month.le(history_end.to_period("M"))).astype("Int8")
    )
del panel_rows, spell_matches, dated_terms, security_terms

## Anticipation baseline · feature table

Each row is an eligible non-member at month-end `mthcaldt` (January 1975 through the membership boundary at December 2022). Features use only data observed at or before that month-end. Return windows require consecutive months with complete returns. Otherwise the feature is missing.

| Feature | Definition |
| --- | --- |
| `log_mktcap` | Log of CRSP security market capitalization at t |
| `mktcap_pct_rank` | Percentile rank of market cap among non-members that month (1 = largest) |
| `dollar_volume_to_mktcap` | Monthly dollar volume (`mthvol × abs(mthprc)`) divided by market cap, a turnover proxy |
| `log_dollar_volume` | Log of monthly dollar volume |
| `mom_12_1` | Compounded return over months t−11 to t−1, skipping month t |
| `ret_1m` | Return in month t |
| `ret_3m` | Compounded return over months t−2 to t |
| `vol_12m` | Standard deviation of monthly returns over months t−11 to t |
| `log_price` | Log of the absolute month-end price |

Each feature except `mktcap_pct_rank` is winsorized at the 1st and 99th percentiles within its month. The percentile rank is already uniform, so it is not winsorized. All features are then standardized to mean 0 and standard deviation 1 within each month.

`y3` and `y6` are labels, not features. They equal 1 when the security's addition becomes effective within 3 or 6 months after t. The labels are right-censored: in the last 3 months of the panel for `y3`, and the last 6 months for `y6`, the horizon extends past the end of the membership data, so the label is missing. **Drop those rows before fitting a model.** Do not treat them as zeros.

In [ ]:
panel = monthly_panel
security = panel["permno"]
month_number = panel["mthcaldt"].dt.year * 12 + panel["mthcaldt"].dt.month
log_return = np.log1p(panel["mthret"])


def months_since(lag):
    return month_number - month_number.groupby(security).shift(lag)


def trailing(values, window, statistic):
    rolled = getattr(values.groupby(security).rolling(window, min_periods=window), statistic)()
    return rolled.droplevel(0).where(months_since(window - 1).eq(window - 1))


price = panel["mthprc"].abs()
dollar_volume = panel["mthvol"] * price
raw_features = pd.DataFrame({
    "permno": security,
    "mthcaldt": panel["mthcaldt"],
    "log_mktcap": np.log(panel["mthcap"].where(panel["mthcap"].gt(0))),
    "dollar_volume_to_mktcap": dollar_volume / (panel["mthcap"] * 1000),  # MthCap is in $ thousands
    "log_dollar_volume": np.log(dollar_volume.where(dollar_volume.gt(0))),
    "mom_12_1": np.expm1(
        trailing(log_return, 11, "sum").groupby(security).shift(1).where(months_since(1).eq(1))
    ),
    "ret_1m": panel["mthret"],
    "ret_3m": np.expm1(trailing(log_return, 3, "sum")),
    "vol_12m": trailing(panel["mthret"], 12, "std"),
    "log_price": np.log(price.where(price.gt(0))),
    "y3": panel["y3"],
    "y6": panel["y6"],
})

features = raw_features.loc[
    ~panel["in_sp500"] & panel["eligible"] & panel["mthcaldt"].between(sample_start, history_end)
].reset_index(drop=True)
features.insert(3, "mktcap_pct_rank", features.groupby("mthcaldt")["log_mktcap"].rank(pct=True))
feature_columns = [
    "log_mktcap", "mktcap_pct_rank", "dollar_volume_to_mktcap", "log_dollar_volume",
    "mom_12_1", "ret_1m", "ret_3m", "vol_12m", "log_price",
]
winsorized_columns = [column for column in feature_columns if column != "mktcap_pct_rank"]

# Unit check for MthVol before standardizing: median monthly dollar volume / market cap.
# If MthVol is in shares, typical monthly turnover is a few percent (roughly 0.01-0.2).
# If it is in hundreds of shares, the median is about 100 times smaller.
turnover_median = features.groupby(features["mthcaldt"].dt.year // 10 * 10)[
    "dollar_volume_to_mktcap"
].median()
print(f"Median dollar volume / market cap, all months: {features['dollar_volume_to_mktcap'].median():.5f}")
print("By decade:")
print(turnover_median.rename_axis("decade").to_string())

by_month = features.groupby("mthcaldt")[winsorized_columns]
winsorized = features[feature_columns].copy()
winsorized[winsorized_columns] = features[winsorized_columns].clip(
    lower=by_month.transform(lambda column: column.quantile(0.01)),
    upper=by_month.transform(lambda column: column.quantile(0.99)),
)
winsorized_by_month = winsorized.groupby(features["mthcaldt"])
features[feature_columns] = (
    (winsorized - winsorized_by_month.transform("mean")) / winsorized_by_month.transform("std")
)
del raw_features, winsorized

print(f"\nRows: {len(features):,}")
print("\nMissing values per column:")
print(features.isna().sum().to_string())
print("\nSummary statistics:")
print(features[feature_columns + ["y3", "y6"]].describe().T.to_string())

## Audit fix 1 · Contaminated events

**What changed.** Cell 12 adds `likely_spin_or_merger`, which flags an addition when its security has no CRSP row in the three calendar months before the effective month. The flag uses only rows dated before the effective month and no information about how long the stock stayed in the index. Flagged events leave the labels and the post-inclusion basket. They stay in `events` and remain index members, so they also stay out of the candidate universe.

**Why.** Every historical addition has `event_type="unknown"`, so spin-offs and merger successors were treated as ordinary additions.

**Effect.** 146 of 1,220 additions are flagged (12.0%). Five additions from February and March 1975 cannot be assessed, because the panel starts too late, and they stay in.

| Decade | Additions | Not assessable | Flagged and excluded |
| --- | --- | --- | --- |
| 1970s | 115 | 5 | 1 |
| 1980s | 253 | 0 | 21 |
| 1990s | 223 | 0 | 30 |
| 2000s | 288 | 0 | 29 |
| 2010s | 237 | 0 | 49 |
| 2020s | 104 | 0 | 16 |

`y3` is unchanged by construction, because a positive row needs a CRSP row in the months before the event. Precision, AUC, and alphas in notebooks 04 and 05 are identical to their previous values. Only the coverage table in notebook 04 moves, from 82.1% to 92.9% in training and from 71.4% to 89.0% in test. That rise comes from the smaller denominator and is not better prediction.

**Limits.** This is a proxy and it does not verify the event type. It can also flag direct IPO additions, moves from other markets, and new PERMNOs from reorganizations. It misses events with CRSP history, such as Bemis's three-day spell in 2019, IPOs less than three months before the event, and acquirers, because the project has no M&A data.